### Kaggle 문제
- https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition

### Import Packages

In [1]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())

CUDA 사용 가능: True


In [2]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0))

GPU 사용 가능: True
Device: NVIDIA GeForce RTX 2060 SUPER


In [3]:
!pip install pandas numpy scikit-learn matplotlib opencv-python pillow requests

In [4]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import numpy as np
from copy import deepcopy
import time

### Setup seed

In [5]:
import random
import os

# device 설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print (device)

seed = 42 # seed 값 설정
random.seed(seed) # 파이썬 난수 생성기 
os.environ['PYTHONHASHSEED'] = str(seed) # 해시 시크릿값 고정
np.random.seed(seed) # 넘파이 난수 생성기 

torch.manual_seed(seed) # 파이토치 CPU 난수 생성기
torch.backends.cudnn.deterministic = True # 확정적 연산 사용 설정
torch.backends.cudnn.benchmark = False   # 벤치마크 기능 사용 해제
torch.backends.cudnn.enabled = False        # cudnn 기능 사용 해제

if device == 'cuda':
    torch.cuda.manual_seed(seed) # 파이토치 GPU 난수 생성기
    torch.cuda.manual_seed_all(seed) # 파이토치 멀티 GPU 난수 생성기

cuda


In [6]:
import os
import zipfile

DATA_PATH = "data/dogsvscats"

os.makedirs(DATA_PATH, exist_ok=True)

# 데이터가 이미 있는지 확인
train_dir = os.path.join(DATA_PATH, "train")
test_dir = os.path.join(DATA_PATH, "test")

if not (os.path.exists(train_dir) and os.path.exists(test_dir)):
    print("데이터를 다운로드하고 있습니다...")
    
    # Kaggle에서 다운로드
    os.system(
        "kaggle competitions download -c dogs-vs-cats-redux-kernels-edition -p data/"
    )

    # 압축 해제
    zip_path = "data/dogs-vs-cats-redux-kernels-edition.zip"

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/")
    
    print("데이터 다운로드 및 압축 해제 완료!")
else:
    print("데이터가 이미 존재합니다. 건너뜁니다.")

데이터가 이미 존재합니다. 건너뜁니다.


### Prepare Data

In [7]:
# train.zip / test.zip 압축 해제
train_zip_path = os.path.join("data", "train.zip")
test_zip_path = os.path.join("data", "test.zip")

train_dir = os.path.join(DATA_PATH, "train")
test_dir = os.path.join(DATA_PATH, "test")

# train.zip 압축 해제
if not os.path.exists(train_dir):
    print("train.zip을 압축 해제하고 있습니다...")
    with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_PATH)
    print("train 폴더 압축 해제 완료!")
else:
    print("train 폴더가 이미 존재합니다. 건너뜁니다.")

# test.zip 압축 해제
if not os.path.exists(test_dir):
    print("test.zip을 압축 해제하고 있습니다...")
    with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_PATH)
    print("test 폴더 압축 해제 완료!")
else:
    print("test 폴더가 이미 존재합니다. 건너뜁니다.")

train 폴더가 이미 존재합니다. 건너뜁니다.
test 폴더가 이미 존재합니다. 건너뜁니다.


In [8]:
# 데이터 경로 수정
import glob
from sklearn.model_selection import train_test_split

train_dir = os.path.join(DATA_PATH, "train")
test_dir = os.path.join(DATA_PATH, "test")

all_train_files = glob.glob(os.path.join(train_dir, "*.jpg"))
test_list = glob.glob(os.path.join(test_dir, "*.jpg"))

train_labels = [
    os.path.basename(path).split('.')[0]
    for path in all_train_files
]

train_list, val_list = train_test_split(
    all_train_files,
    test_size=0.1,
    stratify=train_labels,
    random_state=42
)

print(len(train_list), len(val_list))

22500 2500


In [9]:
train_list[0]

'data/dogsvscats\\train\\cat.4814.jpg'

### Prepare dataset

In [10]:
from torchvision import transforms

input_size = 224
transforms_for_train =  transforms.Compose([
        transforms.RandomResizedCrop(input_size, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

transforms_for_val_test = transforms.Compose([
        transforms.Resize(input_size),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

# CustomDataset 클래스
class CustomDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        
        # 이미지 열기
        img = Image.open(img_path).convert('RGB')
        
        # 변환 적용
        if self.transform is not None:
            img = self.transform(img)
        
        # 레이블 추출 (파일명에서 dog/cat 판단)
        filename = img_path.split('/')[-1].split('\\')[-1]  # Windows 경로 대응
        label_str = filename.split('.')[0]
        
        # 숫자 레이블로 변환 (dog=1, cat=0)
        label = 1 if label_str == 'dog' else 0
        
        return img, label

dataset_train = CustomDataset(train_list, transform=transforms_for_train)
dataset_valid = CustomDataset(val_list, transform=transforms_for_val_test)
dataset_test = CustomDataset(test_list, transform=transforms_for_val_test)

from torch.utils.data import DataLoader # 데이터 로더 클래스

train_batches = DataLoader(dataset=dataset_train, batch_size=8, shuffle=True)
val_batches = DataLoader(dataset=dataset_valid, batch_size=8, shuffle=False)
test_batches = DataLoader(dataset=dataset_test, batch_size=8, shuffle=False)

### Create model

In [11]:
!pip install efficientnet_pytorch==0.7.1

In [12]:
from efficientnet_pytorch import EfficientNet
model = EfficientNet.from_pretrained('efficientnet-b7')
model._fc = nn.Sequential(
    nn.Linear(model._fc.in_features, model._fc.out_features, bias=True),
    nn.LeakyReLU(),
    nn.BatchNorm1d(model._fc.out_features),
    nn.Linear(model._fc.out_features, 1, bias=True),
    nn.Sigmoid()
)
model.to(device)
loss_func = nn.BCELoss()
# optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.001)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.0001)
# optimizer = torch.optim.Adamax(model.parameters(), lr=1e-5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

Loaded pretrained weights for efficientnet-b7


### Define Train Function

In [13]:
def train_model(model, criterion, optimizer, early_stop, epochs, train_loader, valid_loader):
    train_losses, train_accuracies, valid_losses, valid_accuracies, lowest_loss, lowest_epoch = list(), list(), list(), list(), np.inf, 0
    
    for epoch in range(epochs):
        train_loss, train_correct = 0, 0
        valid_loss, valid_correct = 0, 0

        start = time.time()
        model.train()
        for train_x, train_y in train_loader:
            train_x = train_x.to(device)
            train_y = train_y.to(device).float()
            train_y = train_y.view(train_y.size(0), -1)
            
            pred = model(train_x)
            loss = criterion(pred, train_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Sigmoid 출력이므로 0.5 threshold 사용
            y_pred = (pred > 0.5).float()
            train_correct += y_pred.eq(train_y).sum().item()

        train_loss = train_loss / len(train_loader)
        train_losses.append(train_loss)
        train_accuracy = train_correct / len(train_loader.dataset)
        train_accuracies.append(train_accuracy)

        model.eval()
        with torch.no_grad():
            for valid_x, valid_y in valid_loader:
                valid_x = valid_x.to(device)
                valid_y = valid_y.to(device).float()
                valid_y = valid_y.view(valid_y.size(0), -1)
                
                pred = model(valid_x)
                loss = criterion(pred, valid_y)
                valid_loss += loss.item()
            
                # Sigmoid 출력이므로 0.5 threshold 사용
                y_pred = (pred > 0.5).float()
                valid_correct += y_pred.eq(valid_y).sum().item()

        valid_loss = valid_loss / len(valid_loader)
        valid_losses.append(valid_loss)
        valid_accuracy = valid_correct / len(valid_loader.dataset)
        valid_accuracies.append(valid_accuracy)
        
        elapsed_time = time.time() - start
        print(f'[Epoch {epoch+1}/{epochs}]: {elapsed_time:.3f} sec(elapsed time), train loss: {train_losses[-1]:.4f}, train acc: {train_accuracy * 100:.3f}% / valid loss: {valid_losses[-1]:.4f}, valid acc: {valid_accuracy * 100:.3f}%')

        if valid_losses[-1] < lowest_loss:
            lowest_loss = valid_losses[-1]
            lowest_epoch = epoch
            best_model = deepcopy(model.state_dict())
        else:
            if (early_stop > 0) and lowest_epoch + early_stop < epoch:
                print ("Early Stopped", epoch, "epochs")
                break

    model.load_state_dict(best_model)        
    return model, lowest_loss, train_losses, valid_losses, train_accuracies, valid_accuracies



### Training

In [14]:
model, lowest_loss, train_losses, valid_losses, train_accuracies, valid_accuracies = train_model(model, loss_func, optimizer, 0, 10, train_batches, val_batches)

[Epoch 1/10]: 1064.785 sec(elapsed time), train loss: 0.2440, train acc: 89.529% / valid loss: 0.0862, valid acc: 97.880%
[Epoch 2/10]: 1024.777 sec(elapsed time), train loss: 0.1483, train acc: 94.200% / valid loss: 0.0529, valid acc: 98.560%
[Epoch 3/10]: 1024.809 sec(elapsed time), train loss: 0.1117, train acc: 95.929% / valid loss: 0.0394, valid acc: 99.080%
[Epoch 4/10]: 1024.631 sec(elapsed time), train loss: 0.0895, train acc: 97.111% / valid loss: 0.0406, valid acc: 99.000%
[Epoch 5/10]: 1024.703 sec(elapsed time), train loss: 0.0786, train acc: 97.538% / valid loss: 0.0317, valid acc: 99.080%
[Epoch 6/10]: 1024.511 sec(elapsed time), train loss: 0.0616, train acc: 98.071% / valid loss: 0.0261, valid acc: 99.080%
[Epoch 7/10]: 1024.701 sec(elapsed time), train loss: 0.0575, train acc: 98.236% / valid loss: 0.0250, valid acc: 99.080%
[Epoch 8/10]: 1024.767 sec(elapsed time), train loss: 0.0460, train acc: 98.564% / valid loss: 0.0227, valid acc: 99.360%
[Epoch 9/10]: 1027.013 s

In [15]:
torch.save(model.state_dict(), "efficientnet_b7_final.pth")

### Save Model

In [ ]:
PATH = '/content/drive/MyDrive/Colab Notebooks/00_data/dogs-vs-cats/'
torch.save(model.state_dict(), PATH + 'model_efficientnet-b7_without_scheduler_adam_1e5_epoch20.pth')  # 모델 객체의 state_dict 저장

### Load Model

In [ ]:
PATH = '/content/drive/MyDrive/Colab Notebooks/00_data/dogs-vs-cats/'
model.load_state_dict(torch.load(PATH + 'model_efficientnet-b7_without_scheduler_adam_1e5_epoch20.pth'))

<All keys matched successfully>

### Predict & Submit

In [ ]:
test_list = glob.glob(os.path.join(test_dir, '*.jpg'))
dataset_test = CustomDataset(test_list, transform=transforms_for_val_test)
test_batches = DataLoader(dataset=dataset_test, batch_size=8, shuffle=False)

def predict(model, data_loader):
    ids = list()
    with torch.no_grad():
        model.eval()
        ret = None
        for img, fileid in data_loader:
            img = img.to(device)
            pred = model(img)
            ids += list(fileid)
            if ret is None:
                ret = pred.cpu().numpy()
            else:
                ret = np.vstack([ret, pred.cpu().numpy()])
    return ret, ids
   
pred, ids = predict(model, test_batches)

### Submission

In [ ]:
submission = pd.DataFrame({'id': ids, 'label': np.clip(pred, 0.005, 1-0.005).squeeze()})
submission.sort_values(by='id', inplace=True)
submission.reset_index(drop=True, inplace=True)
submission.to_csv('submission.csv', index=False)

### Test for Optimal Clipping

In [ ]:
submission = pd.DataFrame({'id': ids, 'label': np.clip(pred, 0.005, 1-0.005).squeeze()})
submission.sort_values(by='id', inplace=True)
submission.reset_index(drop=True, inplace=True)
submission.to_csv('submission.csv', index=False)